# ChaoSSM — Phase 3a: Selective-SSM Baseline (Implementation & Validation)

## Why this phase

The reviewer objection this project's own manuscript is most exposed to: every result so far
compares chaos against a **non-selective** linear SSM. The `linear` arm used throughout Phases
1-2e has fixed A and B matrices, independent of the input — that makes it structurally closer
to **S4** than to **Mamba/Mamba-2**, whose defining innovation is that the step size, input
projection, and (in the full architecture) output projection are all *functions of the current
input*. Every claim in the paper about "chaos vs. linear SSMs" has, so far, actually been
tested against the *wrong generation* of linear SSM.

This phase adds a genuine selective-SSM baseline and validates it in isolation before
committing to a full task-battery rerun (Phase 3b) — the same cheap-diagnostic-before-big-
commitment pattern used throughout this project (Phase 0 before Phase 1, Phase 2d before
Phase 7-scale claims), justified by the fact that every new mechanism introduced in this
project so far has shipped with at least one real bug that only surfaced under scrutiny.

## Scoping, stated plainly

This is a **matched-simplification baseline**, not a drop-in of the real Mamba-2 codebase —
exactly as the existing `linear` arm is a simplification of S4, not a literal reproduction.
The core mechanism captured is Mamba-2's central innovation: the discretization step size
`Delta_t` and the input matrix `B_t` are both functions of the current input `x_t`, giving the
recurrence input-dependent (selective) dynamics, unlike every arm tested so far. `A` is a
fixed, learnable per-channel decay rate (as in the real Mamba-2 SSD formulation), and the
existing shared readout layer plays the role Mamba-2's `C_t` would, consistent with how the
existing harness already handles output for every other arm.

## This phase's scope

Implement the cell, confirm it trains without divergence, and confirm it *learns* (beats
chance) on the copy task alone — the cheapest, best-understood task in this project's history.
Extending to the full four-task battery, robustness study, and scale sweep against this new
arm is Phase 3b, run only after this validates cleanly.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def setup_results_dir(project="ChaoSSM", phase="phase3a"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase3a


## Configuration

In [2]:
SMOKE_TEST = False   # full run: 4 arms (incl. new mamba2) x 8 seeds, copy task only

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    copy_len: int = 12
    arms: tuple = ("linear", "chaossm_free", "chaossm_corridor", "mamba2")
    seeds: int = 8

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 2
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "copy_len": "12",
  "arms": "('linear', 'chaossm_free', 'chaossm_corridor', 'mamba2')",
  "seeds": "8"
}


## 1. Model — existing arms unchanged, plus the new selective-SSM cell

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    '''Shared wrapper for linear/chaossm arms: gate + fixed A/B + phi(.).'''
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        dtype = self.A.dtype
        return torch.zeros(b, self.N, device=dev, dtype=dtype) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev, dtype=dtype)


class Mamba2Cell(nn.Module):
    #  Matched-simplification selective-SSM baseline (Mamba-2 style), in the same
    #  single-state-vector harness as the other arms. Unlike ChaoSSMCell, Delta_t and B_t
    #  are functions of the CURRENT INPUT -- the property none of the other arms have.
    #
    #  A is a fixed, learnable per-channel decay rate (log-parametrized, standard Mamba
    #  init spread). Discretization: A_bar_t = exp(Delta_t * A). The shared external
    #  readout plays the role Mamba-2's C_t would, matching how every other arm's output
    #  is already handled in this harness.
    def __init__(self, cfg):
        super().__init__()
        self.N = cfg.state_dim
        A_init = torch.arange(1, cfg.state_dim + 1, dtype=torch.float32)
        self.A_log = nn.Parameter(torch.log(A_init))          # A = -exp(A_log) < 0
        self.W_delta = nn.Linear(cfg.d_model, cfg.state_dim)  # input-dependent step size
        self.W_B = nn.Linear(cfg.d_model, cfg.state_dim)      # input-dependent input matrix
        nn.init.zeros_(self.W_delta.bias); nn.init.zeros_(self.W_B.bias)
    def step(self, h, x):
        delta = F.softplus(self.W_delta(x))     # (batch, N) > 0, selective step size
        A = -torch.exp(self.A_log)               # (N,) fixed base decay rate
        A_bar = torch.exp(delta * A)              # (batch, N), selective discretized decay
        B_t = self.W_B(x)                         # (batch, N), selective input term
        return A_bar * h + delta * B_t
    def init_state(self, b, dev):
        return torch.zeros(b, self.N, device=dev, dtype=self.A_log.dtype)


def make_cell(cfg, arm):
    if arm == "mamba2":
        return Mamba2Cell(cfg)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    return ChaoSSMCell(cfg, phi_kind)


class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, arm, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.arm = arm
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = make_cell(cfg, arm)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Parameter-count check

Not an exact match (the arms differ architecturally by design — that's the point), but
worth reporting so the comparison's scope is transparent rather than silently uneven.

In [4]:
def count_params(cfg, arm):
    model = ChaoSSMLM(cfg, arm, cfg.vocab + 2, cfg.vocab + 2)
    cell_params = sum(p.numel() for p in model.cell.parameters())
    total_params = sum(p.numel() for p in model.parameters())
    return cell_params, total_params

PARAM_CHECK = pd.DataFrame([
    dict(arm=arm, cell_params=count_params(CFG, arm)[0], total_params=count_params(CFG, arm)[1])
    for arm in CFG.arms
])
print(PARAM_CHECK.to_string(index=False))
savetable(PARAM_CHECK, "phase3a_param_check.csv")

             arm  cell_params  total_params
          linear         9264         11010
    chaossm_free         9313         11059
chaossm_corridor         9313         11059
          mamba2         4752          6498
saved /content/drive/My Drive/ChaoSSM/results/phase3a/phase3a_param_check.csv


## 3. Estimator (unchanged — only used by chaotic arms)

In [5]:
from torch.func import jacrev

def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

## 4. Task and accuracy — copy, unchanged from every prior phase

In [6]:
def make_copy_batch(cfg, batch, device, gen):
    L, V = cfg.copy_len, cfg.vocab
    sep, pad = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1), sep, device=device),
                     torch.full((batch,L), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1), -100, device=device), src], 1)
    return inp, tgt

VOCAB_SIZE = CFG.vocab + 2
CHANCE = 1.0 / CFG.vocab

def copy_accuracy(logits, tgt):
    mask = tgt != -100
    pred = logits.argmax(-1)
    return ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()

## 5. Trainer

In [7]:
def train_one(arm, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    model = ChaoSSMLM(cfg, arm, VOCAB_SIZE, VOCAB_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    diverged = False
    final_acc = float("nan")
    hist = []

    for step in range(cfg.steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm == "chaossm_free" or arm == "chaossm_corridor")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE)
        if need_traj:
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)
        else:
            lam_val = float("nan")

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % 20 == 0:
            with torch.no_grad():
                final_acc = copy_accuracy(logits, tgt)
            hist.append(dict(step=step, arm=arm, seed=seed, acc=final_acc, gnorm=gn))

    return dict(arm=arm, seed=seed, diverged=diverged, final_acc=final_acc), pd.DataFrame(hist)

## 6. Run — all four arms, copy task

In [8]:
rows, hists = [], []
total = len(CFG.arms) * CFG.seeds
done = 0
for arm in CFG.arms:
    for sd in range(CFG.seeds):
        s, h = train_one(arm, sd, CFG)
        rows.append(s); hists.append(h); done += 1
        flag = " DIVERGED" if s["diverged"] else ""
        print(f"[{done}/{total}] {arm:18s} seed={sd}  final_acc={s['final_acc']:.4f} "
              f"(chance {CHANCE:.4f}){flag}")

SUM = pd.DataFrame(rows); HIST = pd.concat(hists, ignore_index=True)
savetable(SUM, "phase3a_summary.csv"); savetable(HIST, "phase3a_history.csv")
print()
print(SUM.groupby("arm").final_acc.agg(["mean","std"]).round(4).to_string())

[1/32] linear             seed=0  final_acc=0.0742 (chance 0.0625)
[2/32] linear             seed=1  final_acc=0.0573 (chance 0.0625)
[3/32] linear             seed=2  final_acc=0.0885 (chance 0.0625)
[4/32] linear             seed=3  final_acc=0.0469 (chance 0.0625)
[5/32] linear             seed=4  final_acc=0.0521 (chance 0.0625)
[6/32] linear             seed=5  final_acc=0.0573 (chance 0.0625)
[7/32] linear             seed=6  final_acc=0.0703 (chance 0.0625)
[8/32] linear             seed=7  final_acc=0.1133 (chance 0.0625)
[9/32] chaossm_free       seed=0  final_acc=0.1536 (chance 0.0625)
[10/32] chaossm_free       seed=1  final_acc=0.1523 (chance 0.0625)
[11/32] chaossm_free       seed=2  final_acc=0.1289 (chance 0.0625)
[12/32] chaossm_free       seed=3  final_acc=0.1667 (chance 0.0625)
[13/32] chaossm_free       seed=4  final_acc=0.0820 (chance 0.0625)
[14/32] chaossm_free       seed=5  final_acc=0.1068 (chance 0.0625)
[15/32] chaossm_free       seed=6  final_acc=0.1445 (chan

In [9]:
from scipy.stats import wilcoxon

def paired(a, b, col="final_acc"):
    x = SUM[SUM.arm==a].sort_values("seed")[col].values
    y = SUM[SUM.arm==b].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(pair=f"{a} vs {b}", n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(pair=f"{a} vs {b}", n=len(x), p=float(p), median_diff=float(np.median(x-y)))

ST = pd.DataFrame([
    paired("mamba2", "linear"),
    paired("chaossm_free", "mamba2"),
    paired("chaossm_corridor", "mamba2"),
])
print(ST.to_string(index=False))
savetable(ST, "phase3a_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.4f}")

fig, ax = plt.subplots(figsize=(7, 4.2))
means = SUM.groupby("arm").final_acc.mean().reindex(CFG.arms)
stds = SUM.groupby("arm").final_acc.std().reindex(CFG.arms)
ax.bar(range(len(CFG.arms)), means.values, yerr=stds.values, capsize=4,
       color=["gray","tab:blue","tab:green","tab:orange"])
ax.axhline(CHANCE, color="r", ls="--", label="chance")
ax.set_xticks(range(len(CFG.arms))); ax.set_xticklabels(CFG.arms, rotation=15, ha="right")
ax.set(ylabel="copy accuracy", title="Copy task: does chaos beat genuine selectivity, not just S4?")
ax.legend(); ax.grid(alpha=.3, axis="y")
fig.tight_layout(); savefig(fig, "phase3a_copy_comparison.png"); plt.close(fig)

                      pair  n        p  median_diff
          mamba2 vs linear  8 0.406250    -0.001953
    chaossm_free vs mamba2  8 0.007812     0.075521
chaossm_corridor vs mamba2  8 0.007812     0.100260
saved /content/drive/My Drive/ChaoSSM/results/phase3a/phase3a_stats.csv

min achievable two-sided p at n=8: 0.0078
saved /content/drive/My Drive/ChaoSSM/results/phase3a/phase3a_copy_comparison.png


## 7. Verdict

| Observation | Reading |
|---|---|
| `mamba2` trains without divergence and beats chance | The selective-SSM baseline is implemented correctly and ready for Phase 3b's full task battery. |
| `mamba2` matches or beats `chaossm_free` on copying | The advantage chaos showed over the non-selective `linear` arm may be partly or wholly explained by selectivity alone, not chaos specifically — a materially important, humbling finding worth taking directly into Phase 3b rather than treating copying as a settled win. |
| `mamba2` underperforms both chaotic arms, similar to `linear` | Selectivity alone does not explain chaos's copying advantage — strengthens (not just preserves) the original finding, now against a fairer baseline. |
| `mamba2` diverges or fails to learn | A bug in this implementation, not a finding — debug before proceeding to Phase 3b under any circumstances. |

In [10]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    param_check=PARAM_CHECK.to_dict("records"),
    summary=SUM.groupby("arm").final_acc.agg(["mean","std"]).round(4).to_dict("index"),
    stats=ST.to_dict("records"),
    any_diverged=bool(SUM.diverged.any()),
    config=dict(steps=CFG.steps, seeds=CFG.seeds, arms=CFG.arms),
)
savejson(verdict, "phase3a_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase3a/phase3a_verdict.json
{
  "smoke_test": false,
  "param_check": [
    {
      "arm": "linear",
      "cell_params": 9264,
      "total_params": 11010
    },
    {
      "arm": "chaossm_free",
      "cell_params": 9313,
      "total_params": 11059
    },
    {
      "arm": "chaossm_corridor",
      "cell_params": 9313,
      "total_params": 11059
    },
    {
      "arm": "mamba2",
      "cell_params": 4752,
      "total_params": 6498
    }
  ],
  "summary": {
    "chaossm_corridor": {
      "mean": 0.1576,
      "std": 0.0183
    },
    "chaossm_free": {
      "mean": 0.1322,
      "std": 0.028
    },
    "linear": {
      "mean": 0.07,
      "std": 0.0221
    },
    "mamba2": {
      "mean": 0.0599,
      "std": 0.0091
    }
  },
  "stats": [
    {
      "pair": "mamba2 vs linear",
      "n": 8,
      "p": 0.40625,
      "median_diff": -0.001953125
    },
    {
      "pair": "chaossm_free vs mamba2",
      "n": 8,
      "p": 0.00781